# Trajectory Viewer — DB2 unbound T-REMD

Interactive nglview viewer with frame scrubber, speed control, and step size.  
Trajectories in `./trajectories/`: **48 REMD replicas**, NVT equilibration replicas, NPT segments.  
Topology: `DB2_unbound_final_rep000.pdb`

In [2]:
import glob
import os

import MDAnalysis as mda
import nglview as nv
import ipywidgets as widgets
from IPython.display import display, clear_output

TOPOLOGY = "./nvt/rep000/nvt.tpr"
TRAJ_DIR = "./analysis/"

remd = sorted(glob.glob(f"{TRAJ_DIR}/remd_rep*.xtc"))
nvt  = sorted(glob.glob(f"{TRAJ_DIR}/nvt_rep*.xtc"))
npt  = sorted(glob.glob(f"{TRAJ_DIR}/DB2_unbound_npt_seg*.xtc"))

traj_options = (
    [(f"remd  rep{i:03d}", p) for i, p in enumerate(remd)] +
    [(f"nvt   rep{i:03d}", p) for i, p in enumerate(nvt)]  +
    [(f"npt   seg{i+1}",   p) for i, p in enumerate(npt)]
)

print(f"Discovered: {len(remd)} REMD | {len(nvt)} NVT | {len(npt)} NPT trajectories")

Discovered: 1 REMD | 0 NVT | 0 NPT trajectories


In [4]:
def launch_viewer(traj_path):
    u = mda.Universe(TOPOLOGY, traj_path)
    view = nv.show_mdanalysis(u)
    n = u.trajectory.n_frames

    # Play/pause widget — drives frame scrubber via jslink (all client-side)
    play = widgets.Play(
        value=0, min=0, max=n - 1,
        step=1, interval=100,
        description="▶",
        layout=widgets.Layout(width="100px"),
    )

    frame_slider = widgets.IntSlider(
        value=0, min=0, max=n - 1, step=1,
        description="Frame:",
        continuous_update=True,
        layout=widgets.Layout(width="580px"),
        style={"description_width": "60px"},
    )
    speed_slider = widgets.IntSlider(
        value=100, min=10, max=2000, step=10,
        description="Delay (ms):",
        layout=widgets.Layout(width="400px"),
        style={"description_width": "80px"},
    )
    step_slider = widgets.IntSlider(
        value=1, min=1, max=50, step=1,
        description="Step:",
        layout=widgets.Layout(width="400px"),
        style={"description_width": "60px"},
    )

    # Client-side links: play ↔ frame scrubber ↔ nglview frame
    widgets.jslink((play, "value"),        (frame_slider, "value"))
    widgets.jslink((frame_slider, "value"), (view, "frame"))

    # Speed and step feed directly into the Play widget
    widgets.jslink((speed_slider, "value"), (play, "interval"))
    widgets.jslink((step_slider,  "value"), (play, "step"))

    label = widgets.HTML(
        f"<b>{os.path.basename(traj_path)}</b> &nbsp;|&nbsp; {n} frames"
    )
    controls = widgets.VBox(
        [label,
         widgets.HBox([play, frame_slider]),
         widgets.HBox([speed_slider, step_slider])],
        layout=widgets.Layout(margin="8px 0 0 0"),
    )
    return view, controls


# ── Trajectory selector ──────────────────────────────────────────────────────
dropdown = widgets.Dropdown(
    options=traj_options,
    description="Trajectory:",
    layout=widgets.Layout(width="440px"),
    style={"description_width": "90px"},
)

out = widgets.Output()


def _on_select(change):
    if change["name"] != "value":
        return
    with out:
        clear_output(wait=True)
        view, controls = launch_viewer(change["new"])
        display(view)
        display(controls)


dropdown.observe(_on_select, names="value")

# Initial load
with out:
    _view0, _ctrl0 = launch_viewer(dropdown.value)
    display(_view0)
    display(_ctrl0)

display(widgets.VBox([dropdown, out]))